# Making an Ingest Tool for multiple HTML files

First thing is to make a notebook called 01_ingest_jesuit_relations_html.ipynb. So that's this notebook.

In [ ]:
# import the necessary packages and stuff 

from pathlib import Path
import re
import json

from bs4 import BeautifulSoup


## The basic idea
The basic idea is to iterate through the HTML files and to make them into a big json file, where the arrays are the paragraphs in the html. 

In [4]:
# CHANGE THIS if your directory name is different
HTML_DIR = Path("jesuit_relations_html")

assert HTML_DIR.exists(), f"{HTML_DIR} not found"

Next, insert some code to test that we have the correct files and can find the html. 

In [5]:
html_files = sorted([
    p for p in HTML_DIR.glob("*.html")
    if p.name != "index.html"
])

len(html_files), html_files[:5]

(71,
 [PosixPath('jesuit_relations_html/relations_01.html'),
  PosixPath('jesuit_relations_html/relations_02.html'),
  PosixPath('jesuit_relations_html/relations_03.html'),
  PosixPath('jesuit_relations_html/relations_04.html'),
  PosixPath('jesuit_relations_html/relations_05.html')])

Ingest a single file to test and make sure. Inspect the HTML structure. Note that the page numbers are wrapped in \<B>\ tags. 

In [7]:
sample_file = html_files[0]

with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
    raw_html = f.read()

raw_html[:10000]

'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2//EN">\n<HTML>\n<HEAD>\n<META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=windows-1252">\n<META NAME="Generator" CONTENT="Microsoft Word 97">\n<TITLE>The Jesuit Relations and Allied Documents Volume 1</TITLE>\n</HEAD>\n<BODY bgcolor="ffffff">\n\n<CENTER><FONT SIZE=5> The   Jesuit   Relations   and   Allied   Documents</P>\n  Travels and Explorations</P>\n of the Jesuit Missionaries</P>\n in New France</P>\n </P>\n 16101791</P></CENTER>\n \n <BR>\n<BR>\n<BR>\n\n<CENTER><font size=4> THE ORIGINAL FRENCH, LATIN, AND ITALI-</P>\n IAN    TEXTS,    WITH    ENGLISH    TRANSLA-</P>\n TIONS    AND    NOTES;    ILLUSTRATED    BY</P>\n PORTRAITS,   MAPS,   AND   FACSIMILES</P></center>\n \n<P ALIGN="CENTER">&nbsp;</P></DIR></CENTER>\n</DIR>\n</DIR>\n</DIR>\n\n</FONT><FONT SIZE=1><P ALIGN="CENTER">EDITED BY</P>\n</FONT><FONT SIZE=1><P ALIGN="CENTER">Reuben Gold Thwaites</P>\n</FONT><FONT SIZE=1><P ALIGN="CENTER">Secretary of the State historica

## Now parse the file. 

Use beautiful soup to get started. 

In [ ]:
soup = BeautifulSoup(raw_html, "html.parser")

Now inspect

In [13]:
set(tag.name for tag in soup.find_all())

{'b',
 'blockquote',
 'body',
 'br',
 'center',
 'dir',
 'font',
 'head',
 'html',
 'i',
 'img',
 'li',
 'meta',
 'ol',
 'p',
 'table',
 'td',
 'title',
 'tr'}

## Now identify paragraphs.

Paragraphs are the atomic units of this json structure, so let's find them in the html files. 

Note that there has to be a good way -- tbd-- to cut off the front matter of each volume, including TOC and title page and everything. Here in the script below we display the paragraphs starting at 100-105, because that's real paragraph text. I need a method to identify in each html file where the real text begins. 

In [20]:
paragraphs = []

for p in soup.find_all("p"):
    text = p.get_text(separator=" ", strip=True)
    if text:
        paragraphs.append(text)

len(paragraphs), paragraphs[100:105]

(867,
 ['Father Gabriel Druillettes, of the Jesuit mission at Sillery, near Québec, went to the Kennebec country in 1646, invited thither by converted Abenakis who had been at Sillery, and during visits, extending through a period of eleven years, was more than ordinarily successful in the task of gaining Indian converts to Christianity. In 1650, he made a notable visit to the Puritans of Eastern Massachusetts, during which was discussed the proposed union between New France and New England, against the Iroquois. Upon the final departure of Druillettes in r657, the Abenakis were but spasmodically served with missionaries; occasionally a Jesuit appeared among them, but the field could not be persistently worked, owing to the demands upon the order from other [page 14] quarters. The fathers now sought to draw Abenaki converts to Sillery, and later to St. Francis de Sales, at the falls of the Chaudiere, which soon became almost exclusively an Abenaki mission.',
  'In 1688, Father Bigot, o